# ODE PINN Project

The aim of this project is to reinforce the theory on PINN's by building a basic PINN. For this project, we will consider the IVP given by the following differential equation:
$$ u'(t) + u(t) = 0, \hspace{3mm} u(0)=1, \hspace{3mm} t\in[0,5]$$

This project acts as a stepping stone in our learning and serves as a simplified learning model which will be useful when we tackle the next project of trying to build a neural network that solves the heat equation.

In [79]:
import torch
import matplotlib.pyplot as plt
import torch.nn as nn

## First step

We are going to use the test polynomial $v(t)=t^3-2t+1$ (since this is an easily differentiable analytic function) as a basic means of checking the computation of the derivative that is computed using PyTorch.

In [80]:
# Initialising data type and seed
torch.set_default_dtype(torch.float64)
torch.manual_seed(0)

# Building 6 evaluation points in the domain and storing them as a column vector

t = torch.linspace(0.0, 5.0, 6).reshape(-1, 1)
t.requires_grad_(True)

# Test polynomial
v = t**3 - 2*t + 1

print(t.shape)

torch.Size([6, 1])


In [81]:
# Computing derivative using PyTorch

dv_dt = torch.autograd.grad(
    outputs=v,
    inputs=t,
    grad_outputs=torch.ones_like(v),
    create_graph=True,)[0]

A brief note on the above function. torch.autgrad.grad() is the usual PyTorch function which computes the gradient through the neural network. Setting 'outputs=v' initialises the output as the test polynomial $v$, 'inputs=t' initialises the input as $t$ so that collectively they produce the gradient of $v$ with respect to $t$. Additionally, 'grad_outputs=torch.ones_like(v)' creates a tensor the same size as $v$ with all entries 1 to serve as the first term in the chain rule when calculating the following adjoints. Namely, it represents $\frac{\partial\mathcal{L}}{\partial v}=1$ but in the same tensor form as $v$ to ensure appropriate matrix multiplication. Finally, 'create_graph=True' keeps a differentiable computation of the derivative to store the gradient data, whilst still allowing further differentiate through it. This is particularly important since for when we train our PINN. Namely, 'create_graph=True' will allow the parameter gradient to pass through the computed input derivative. 

In [82]:
# Computing the exact derivative (analytic function)

dv_dt_exact = 3*t**2 - 2

# Maximum discrepancy between analytic and computed derivative

max_disc = torch.max(torch.abs(dv_dt-dv_dt_exact))

print("Maximum discrepancy between analytic and computed derivative is: " + str(max_disc))

# Decay equation residual

rv = dv_dt + v

Maximum discrepancy between analytic and computed derivative is: tensor(0., grad_fn=<MaxBackward1>)


## PINN Model

We will use the same architecture that we have used in the previous project. Namely an input, 2 hidden layers and an output with the form:

$$ 1 \rightarrow 16 \rightarrow 16 \rightarrow 1 $$

with $\tanh$ after each hidden layer and an affine output layer. We are going to use the torch.nn module which has a few more features that will be convenient for our use.

In [87]:
# Creating the model using torch.nn. Sequential ensures the layers are constructed in the same way they are coded.
# Writing it this way as opposed to a function ensures we use the same trainable paramaters each time, so that the
# optimiser can improve these during training.

torch.manual_seed(0)

model = nn.Sequential(
    nn.Linear(1, 16),
    nn.Tanh(),
    nn.Linear(16, 16),
    nn.Tanh(),
    nn.Linear(16, 1),
) 

u_pred = model(t)

# Computing the gradient of the model

du_dt = torch.autograd.grad(
    outputs=u_pred,
    inputs=t,
    grad_outputs=torch.ones_like(u_pred),
    create_graph=True,)[0]

# Paramater residuals

r_eq = du_dt + u_pred

# Loss of the paramaters

L_eq = torch.mean(r_eq**2)

# Computing the initial input. We will simply call the model on our initial condition t_0=0

t0 = torch.zeros((1,1))
u0 = model(t0)

# Loss of the initial condition. We compute the mean so that it returns a scalar, same as L_eq.

L_init = ((u0-1)**2).mean()